# Validação causal dos parâmetros $\theta$ — versão standalone

Este notebook **não depende do `01_generate_dataset.ipynb`** e **não executa novamente as 10.000 otimizações**.

Ele reconstrói sozinho:

1. os dados financeiros a partir de:

```python
pd.read_csv(
    "../data/assets/stocks_price.csv",
    index_col=0,
)
```

2. a mesma função clássica de portfólio:

$$
f(x)
=
q\,x^\top\Sigma x
-
(1-q)\,\mu^\top x,
$$

com:

$$
\sum_i x_i = 4,
\qquad
x_i\in\{0,1\};
$$

3. a solução clássica por CVXPY, com enumeração das 210 carteiras como auditoria;
4. o mesmo QUBO e o mesmo Hamiltoniano de Ising;
5. as portas parametrizadas `CY` e `CCY`;
6. o mesmo `DickeStateAnsatz(10, 4)` customizado;
7. a ordem dos 30 parâmetros;
8. a validação causal sobre o banco já salvo.

O banco existente é apenas lido por `pd.read_pickle`. Não há `COBYLA.minimize`, `safe_optimize`, `TOTAL_RUNS` ou laço de geração neste notebook.


## Lógica científica

O teste mantém o Hamiltoniano e o ansatz fixos e usa as soluções já existentes para:

- localizar regiões angulares associadas a menor energia e maior recuperação do bitstring ótimo;
- separar descoberta e validação;
- escolher controles negativos;
- aplicar intervenções locais nos parâmetros;
- medir necessidade e suficiência local;
- avaliar as intervenções com `Statevector`, sem shots.

Os parâmetros candidatos continuam sendo:

$$
\theta_2,\quad
\theta_3,\quad
\theta_{14},\quad
\theta_{17}.
$$

A reconstrução precisa coincidir com o banco original. Por isso, antes da análise causal, o notebook confere:

- número de ativos;
- quantidade de parâmetros;
- energia exata;
- bitstring ótimo;
- vetor de retornos;
- matriz de covariância;
- ordem estrutural dos parâmetros.


In [ ]:
# ============================================================
# 1. IMPORTS E CONFIGURAÇÃO
# ============================================================

from itertools import combinations
from pathlib import Path
import ast
import json
import warnings

import cvxpy as cp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scipy.stats import wilcoxon

from docplex.mp.model import Model

from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector

from qiskit_optimization.translators import (
    from_docplex_mp,
)

from qiskit_optimization.converters import (
    QuadraticProgramToQubo,
)

try:
    from qiskit_algorithms import (
        NumPyMinimumEigensolver,
    )
except ImportError:
    from qiskit.algorithms import (
        NumPyMinimumEigensolver,
    )


# ------------------------------------------------------------
# PROBLEMA ORIGINAL
# ------------------------------------------------------------

# Caminho solicitado pelo usuário.
STOCK_PATH = Path(
    "../data/assets/stocks_price.csv"
)

N_ASSETS = 10
K = 4
Q = 0.5

# A semente é usada somente para reproduzir os nomes simbólicos
# das portas e para separar descoberta/validação.
RANDOM_STATE = 42
PARAMETER_NAME_SEED = 42

# Quando True, qualquer incompatibilidade entre a reconstrução
# e o banco interrompe o notebook antes das intervenções.
STRICT_RECONSTRUCTION = True


# ------------------------------------------------------------
# BANCO JÁ EXISTENTE
# ------------------------------------------------------------

# Preencha com um caminho específico somente quando necessário.
# Quando permanece None, o código procura nos locais abaixo.
MERGED_PATH_OVERRIDE = None

MERGED_CANDIDATES = [
    Path("results/merged.pkl"),
    Path("../results/merged.pkl"),
    Path("results/vqe_dataset_combined.pkl"),
    Path("../results/vqe_dataset_combined.pkl"),
]


# ------------------------------------------------------------
# VALIDAÇÃO CAUSAL
# ------------------------------------------------------------

CANDIDATE_THETAS = [
    2,
    3,
    14,
    17,
]

N_CONTROL_THETAS = 2

DISCOVERY_FRACTION = 0.70

N_ANGLE_BINS = 24
MIN_RUNS_PER_BIN = 30

N_TEST_VECTORS = 20

CAUSAL_OUTPUT = Path(
    "results/causal_validation_standalone"
)

FIGURE_OUTPUT = (
    CAUSAL_OUTPUT
    / "figures"
)

TABLE_OUTPUT = (
    CAUSAL_OUTPUT
    / "tables"
)

FIGURE_OUTPUT.mkdir(
    parents=True,
    exist_ok=True,
)

TABLE_OUTPUT.mkdir(
    parents=True,
    exist_ok=True,
)

rng = np.random.default_rng(
    RANDOM_STATE
)

plt.rcParams.update(
    {
        "figure.figsize": (
            12,
            7,
        ),
        "figure.dpi": 120,
        "axes.titlesize": 15,
        "axes.labelsize": 12,
        "font.size": 11,
    }
)

print(
    "Modo standalone configurado."
)

print(
    "Nenhuma otimização COBYLA será executada."
)


## 2. Carregar preços e banco salvo

O CSV reconstrói o problema físico. O `merged.pkl` fornece somente as execuções já produzidas.

Quando as colunas `assets_return` e `covariance` estão presentes no banco, elas são usadas para conferir automaticamente qual normalização dos retornos foi empregada no experimento original.


In [ ]:
# ============================================================
# 2. CARREGAR O CSV E O BANCO EXISTENTE
# ============================================================

if not STOCK_PATH.is_file():
    raise FileNotFoundError(
        "O CSV não foi encontrado em: "
        f"{STOCK_PATH.resolve()}"
    )


stocks_prices = pd.read_csv(
    "../data/assets/stocks_price.csv",
    index_col=0,
)

stocks_prices = (
    stocks_prices
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
    .ffill()
    .bfill()
    .dropna(
        axis=1,
        how="all",
    )
    .dropna(
        axis=0,
        how="any",
    )
)

if (
    stocks_prices.shape[1]
    != N_ASSETS
):
    raise ValueError(
        f"Eram esperados {N_ASSETS} ativos, "
        f"mas o CSV possui {stocks_prices.shape[1]}."
    )

period_returns = (
    stocks_prices
    .pct_change(
        fill_method=None
    )
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
    .dropna()
)

if len(period_returns) < 2:
    raise ValueError(
        "O CSV não possui observações suficientes "
        "para calcular a covariância."
    )


if MERGED_PATH_OVERRIDE is not None:
    MERGED_PATH = Path(
        MERGED_PATH_OVERRIDE
    )
else:
    MERGED_PATH = next(
        (
            candidate
            for candidate
            in MERGED_CANDIDATES
            if candidate.is_file()
        ),
        None,
    )

if MERGED_PATH is None:
    checked_paths = "\n".join(
        f" - {candidate.resolve()}"
        for candidate
        in MERGED_CANDIDATES
    )

    raise FileNotFoundError(
        "Nenhum banco consolidado foi encontrado.\n"
        "Caminhos verificados:\n"
        f"{checked_paths}"
    )

dataset = pd.read_pickle(
    MERGED_PATH
).copy()

required_columns = [
    "best_parameters",
    "objective_function_value",
    "most_frequent_bitstring",
]

missing_columns = [
    column
    for column
    in required_columns
    if column not in dataset.columns
]

if missing_columns:
    raise KeyError(
        "Colunas ausentes no banco: "
        + ", ".join(
            missing_columns
        )
    )

print(
    "CSV:",
    STOCK_PATH.resolve(),
)

print(
    "Banco:",
    MERGED_PATH.resolve(),
)

print(
    "Linhas carregadas:",
    len(dataset),
)

print(
    "Dimensão dos preços:",
    stocks_prices.shape,
)

print(
    "Dimensão dos retornos:",
    period_returns.shape,
)


## 3. Reconstruir $\mu$ e $\Sigma$

O notebook calcula algumas convenções comuns e compara cada uma com os valores registrados no banco.

A escolha automática serve para evitar que uma diferença de escala — por exemplo, anualização por 252 — altere a energia reconstruída.

Quando o banco não possui os metadados necessários, a convenção padrão é:

```text
retorno médio por período
covariância por período
```


In [ ]:
# ============================================================
# 3. RECONSTRUÇÃO DOS RETORNOS E DA COVARIÂNCIA
# ============================================================

raw_mean = (
    period_returns
    .mean(axis=0)
    .to_numpy(
        dtype=float
    )
)

raw_sum = (
    period_returns
    .sum(axis=0)
    .to_numpy(
        dtype=float
    )
)

raw_covariance = (
    period_returns
    .cov()
    .to_numpy(
        dtype=float
    )
)

raw_covariance = 0.5 * (
    raw_covariance
    + raw_covariance.T
)


def make_psd(
    matrix,
):
    """
    Remove somente autovalores negativos produzidos
    por erro numérico.
    """
    matrix = np.asarray(
        matrix,
        dtype=float,
    )

    matrix = 0.5 * (
        matrix
        + matrix.T
    )

    eigenvalues, eigenvectors = np.linalg.eigh(
        matrix
    )

    eigenvalues = np.clip(
        eigenvalues,
        0.0,
        None,
    )

    corrected = (
        eigenvectors
        @ np.diag(
            eigenvalues
        )
        @ eigenvectors.T
    )

    return 0.5 * (
        corrected
        + corrected.T
    )


candidate_constructions = {
    "mean_raw": (
        raw_mean,
        raw_covariance,
    ),
    "mean_annualized_252": (
        252.0
        * raw_mean,
        252.0
        * raw_covariance,
    ),
    "sum_return_raw_covariance": (
        raw_sum,
        raw_covariance,
    ),
}


def vector_relative_error(
    reconstructed,
    reference,
):
    reconstructed = np.asarray(
        reconstructed,
        dtype=float,
    ).reshape(-1)

    reference = np.asarray(
        reference,
        dtype=float,
    ).reshape(-1)

    denominator = max(
        np.linalg.norm(
            reference
        ),
        1e-15,
    )

    return float(
        np.linalg.norm(
            reconstructed
            - reference
        )
        / denominator
    )


bank_has_financial_metadata = (
    "assets_return"
    in dataset.columns
    and "covariance"
    in dataset.columns
)

construction_audit_rows = []

if bank_has_financial_metadata:
    bank_assets_return = np.asarray(
        dataset[
            "assets_return"
        ].iloc[0],
        dtype=float,
    ).reshape(-1)

    bank_covariance = np.asarray(
        dataset[
            "covariance"
        ].iloc[0],
        dtype=float,
    )

    for construction_name, (
        candidate_return,
        candidate_covariance,
    ) in candidate_constructions.items():
        return_error = vector_relative_error(
            candidate_return,
            bank_assets_return,
        )

        covariance_error = vector_relative_error(
            candidate_covariance,
            bank_covariance,
        )

        construction_audit_rows.append(
            {
                "construction": construction_name,
                "return_relative_error": return_error,
                "covariance_relative_error": covariance_error,
                "combined_error": (
                    return_error
                    + covariance_error
                ),
            }
        )

    construction_audit_df = (
        pd.DataFrame(
            construction_audit_rows
        )
        .sort_values(
            "combined_error"
        )
        .reset_index(
            drop=True
        )
    )

    display(
        construction_audit_df
    )

    selected_construction = str(
        construction_audit_df[
            "construction"
        ].iloc[0]
    )

    best_combined_error = float(
        construction_audit_df[
            "combined_error"
        ].iloc[0]
    )

    if best_combined_error <= 1e-7:
        assets_return, covariance = (
            candidate_constructions[
                selected_construction
            ]
        )

        reconstruction_source = (
            "csv_exact_match"
        )
    else:
        warnings.warn(
            "Nenhuma convenção calculada do CSV reproduziu "
            "exatamente os metadados do banco. "
            "Os valores salvos no próprio banco serão usados "
            "para preservar o Hamiltoniano original."
        )

        assets_return = (
            bank_assets_return.copy()
        )

        covariance = (
            bank_covariance.copy()
        )

        selected_construction = (
            "bank_metadata_fallback"
        )

        reconstruction_source = (
            "bank_metadata_fallback"
        )
else:
    assets_return, covariance = (
        candidate_constructions[
            "mean_raw"
        ]
    )

    selected_construction = (
        "mean_raw"
    )

    reconstruction_source = (
        "csv_without_bank_metadata"
    )

covariance = make_psd(
    covariance
)

assets_return = np.asarray(
    assets_return,
    dtype=float,
).reshape(-1)

tickers = list(
    stocks_prices.columns
)

print(
    "Construção selecionada:",
    selected_construction,
)

print(
    "Fonte final:",
    reconstruction_source,
)

print(
    "Retornos:",
    assets_return.shape,
)

print(
    "Covariância:",
    covariance.shape,
)

print(
    "Menor autovalor:",
    np.linalg.eigvalsh(
        covariance
    ).min(),
)


## 4. CVXPY, enumeração, QUBO e Ising

A função clássica usada em todas as etapas é exatamente:

$$
f(x)
=
0.5\,x^\top\Sigma x
-
0.5\,\mu^\top x.
$$

A enumeração das:

$$
\binom{10}{4}
=
210
$$

carteiras garante uma referência exata mesmo quando nenhum solver MIQP está instalado no CVXPY.


In [ ]:
# ============================================================
# 4. CVXPY E VERIFICAÇÃO EXATA DAS 210 CARTEIRAS
# ============================================================

x_cvx = cp.Variable(
    N_ASSETS,
    boolean=True,
    name="portfolio_selection",
)

cvx_objective = cp.Minimize(
    Q
    * cp.quad_form(
        x_cvx,
        cp.psd_wrap(
            covariance
        ),
    )
    - (
        1.0
        - Q
    )
    * assets_return
    @ x_cvx
)

cvx_constraints = [
    cp.sum(
        x_cvx
    )
    == K,
]

cvx_problem = cp.Problem(
    cvx_objective,
    cvx_constraints,
)

miqp_priority = [
    "GUROBI",
    "CPLEX",
    "MOSEK",
    "SCIP",
    "XPRESS",
    "COPT",
    "ECOS_BB",
]

installed_solvers = set(
    cp.installed_solvers()
)

selected_solver = next(
    (
        solver
        for solver
        in miqp_priority
        if solver
        in installed_solvers
    ),
    None,
)

cvxpy_solution = None
cvxpy_value = None
cvxpy_status = (
    "solver_miqp_indisponivel"
)

if selected_solver is not None:
    try:
        cvx_problem.solve(
            solver=selected_solver,
            verbose=False,
        )

        cvxpy_status = (
            cvx_problem.status
        )

        if x_cvx.value is not None:
            cvxpy_solution = np.rint(
                np.asarray(
                    x_cvx.value
                ).reshape(-1)
            ).astype(int)

            cvxpy_value = float(
                cvx_problem.value
            )
    except Exception as exc:
        warnings.warn(
            "O CVXPY não concluiu a solução MIQP: "
            f"{type(exc).__name__}: {exc}"
        )


def classical_objective(
    x_binary,
):
    x_binary = np.asarray(
        x_binary,
        dtype=float,
    ).reshape(-1)

    return float(
        Q
        * x_binary
        @ covariance
        @ x_binary
        - (
            1.0
            - Q
        )
        * assets_return
        @ x_binary
    )


enumeration_rows = []

for selected_indices in combinations(
    range(
        N_ASSETS
    ),
    K,
):
    candidate = np.zeros(
        N_ASSETS,
        dtype=int,
    )

    candidate[
        list(
            selected_indices
        )
    ] = 1

    bitstring_asset_order = "".join(
        str(
            int(value)
        )
        for value
        in candidate
    )

    bitstring_qiskit_order = (
        bitstring_asset_order[
            ::-1
        ]
    )

    enumeration_rows.append(
        {
            "selected_indices": (
                selected_indices
            ),
            "x_asset_order": (
                candidate
            ),
            "bitstring_asset_order": (
                bitstring_asset_order
            ),
            "bitstring_qiskit_order": (
                bitstring_qiskit_order
            ),
            "objective": (
                classical_objective(
                    candidate
                )
            ),
        }
    )

enumeration_df = (
    pd.DataFrame(
        enumeration_rows
    )
    .sort_values(
        "objective"
    )
    .reset_index(
        drop=True
    )
)

enumeration_best = (
    enumeration_df.iloc[0]
)

ENUMERATION_ENERGY = float(
    enumeration_best[
        "objective"
    ]
)

ENUMERATION_BITSTRING = str(
    enumeration_best[
        "bitstring_qiskit_order"
    ]
)

print(
    "Solver CVXPY:",
    selected_solver,
)

print(
    "Status CVXPY:",
    cvxpy_status,
)

print(
    "Energia por enumeração:",
    ENUMERATION_ENERGY,
)

print(
    "Bitstring por enumeração:",
    ENUMERATION_BITSTRING,
)

display(
    enumeration_df[
        [
            "selected_indices",
            "bitstring_asset_order",
            "bitstring_qiskit_order",
            "objective",
        ]
    ].head(
        10
    )
)

if cvxpy_solution is not None:
    cvxpy_bitstring_qiskit = "".join(
        str(
            int(value)
        )
        for value
        in cvxpy_solution[
            ::-1
        ]
    )

    print(
        "Bitstring CVXPY:",
        cvxpy_bitstring_qiskit,
    )

    print(
        "Objetivo CVXPY:",
        cvxpy_value,
    )


In [ ]:
# ============================================================
# 5. MESMA FUNÇÃO CLÁSSICA -> QUBO -> ISING
# ============================================================

portfolio_model = Model(
    name="portfolio_original"
)

x_docplex = [
    portfolio_model.binary_var(
        name=f"x_{index}"
    )
    for index
    in range(
        N_ASSETS
    )
]

risk_expression = portfolio_model.sum(
    float(
        covariance[
            row,
            column,
        ]
    )
    * x_docplex[
        row
    ]
    * x_docplex[
        column
    ]
    for row
    in range(
        N_ASSETS
    )
    for column
    in range(
        N_ASSETS
    )
)

return_expression = portfolio_model.sum(
    float(
        assets_return[
            index
        ]
    )
    * x_docplex[
        index
    ]
    for index
    in range(
        N_ASSETS
    )
)

portfolio_model.minimize(
    Q
    * risk_expression
    - (
        1.0
        - Q
    )
    * return_expression
)

portfolio_model.add_constraint(
    portfolio_model.sum(
        x_docplex
    )
    == K,
    ctname="budget",
)

quadratic_program = from_docplex_mp(
    portfolio_model
)

qubo = QuadraticProgramToQubo().convert(
    quadratic_program
)

ising, offset = qubo.to_ising()

exact_solver = NumPyMinimumEigensolver()

exact_result = (
    exact_solver
    .compute_minimum_eigenvalue(
        operator=ising
    )
)

EXACT_ENERGY = float(
    np.real(
        exact_result.eigenvalue
        + offset
    )
)

exact_state_dict = (
    exact_result
    .eigenstate
    .to_dict()
)

OPTIMAL_BITSTRINGS = sorted(
    [
        str(
            bitstring
        ).replace(
            " ",
            "",
        )
        for bitstring, amplitude
        in exact_state_dict.items()
        if abs(
            amplitude
        )
        > 1e-12
    ]
)

EXACT_BITSTRING = (
    OPTIMAL_BITSTRINGS[0]
)

print(
    "Qubits do Ising:",
    ising.num_qubits,
)

print(
    "Offset:",
    offset,
)

print(
    "Energia exata:",
    EXACT_ENERGY,
)

print(
    "Bitstring(s) ótimo(s):",
    OPTIMAL_BITSTRINGS,
)

energy_difference_enumeration = abs(
    EXACT_ENERGY
    - ENUMERATION_ENERGY
)

if (
    energy_difference_enumeration
    > 1e-8
):
    raise RuntimeError(
        "A energia do Ising não coincide com a enumeração: "
        f"diferença={energy_difference_enumeration:.3e}"
    )


## 5. `DickeStateAnsatz` embutido

As funções abaixo são colocadas diretamente neste notebook.

Assim, a validação não precisa importar a classe de outro arquivo ou executar uma célula do notebook de geração.


In [ ]:
# ============================================================
# 6. PORTAS CY, CCY E DickeStateAnsatz
# ============================================================

def CY_parameterized(
    identifier,
):
    """
    Rotação Y controlada usada no código original.

    O identificador cria um nome simbólico único para o theta.
    O valor numérico só entra depois em assign_parameters.
    """
    parameter = ParameterVector(
        name=f"x{identifier}",
        length=1,
    )

    circuit = QuantumCircuit(
        2
    )

    circuit.cry(
        parameter[0],
        1,
        0,
    )

    return circuit.to_gate(
        label="CY"
    )


def CCY_parameterized(
    identifier,
):
    """
    Porta de três qubits parametrizada usada no código original.
    """
    parameter = ParameterVector(
        name=f"y{identifier}",
        length=1,
    )

    circuit = QuantumCircuit(
        3
    )

    circuit.ry(
        parameter[0],
        0,
    )

    circuit.ccx(
        2,
        1,
        0,
    )

    circuit.ry(
        -parameter[0],
        0,
    )

    circuit.ccx(
        2,
        1,
        0,
    )

    return circuit.to_gate(
        label="CCY"
    )


def DickeStateAnsatz(
    n,
    k,
):
    """
    Implementação customizada usada no experimento original.

    Os últimos k qubits recebem X. As portas CY e CCY
    redistribuem amplitudes dentro do setor de k excitações.
    """
    quantum_register = QuantumRegister(
        n,
        "q",
    )

    circuit = QuantumCircuit(
        quantum_register
    )

    # Estado inicial com exatamente k excitações.
    for excitation_index in range(
        k
    ):
        circuit.x(
            n
            - excitation_index
            - 1
        )

    auxiliary_counter = 1

    for l_index in range(
        n
    )[::-1]:
        for i_index in range(
            l_index - 1,
            l_index - 1 - k,
            -1,
        ):
            if i_index >= 0:
                identifier = (
                    str(
                        l_index
                    )
                    + str(
                        i_index
                    )
                    + str(
                        auxiliary_counter
                    )
                    + str(
                        np.random.randint(
                            0,
                            int(
                                1e8
                            ),
                        )
                    )
                )

                if (
                    i_index
                    == l_index - 1
                ):
                    circuit.cx(
                        quantum_register[
                            i_index
                        ],
                        quantum_register[
                            l_index
                        ],
                    )

                    circuit.append(
                        CY_parameterized(
                            identifier
                        ),
                        [
                            quantum_register[
                                i_index
                            ],
                            quantum_register[
                                l_index
                            ],
                        ],
                    )

                    circuit.cx(
                        quantum_register[
                            i_index
                        ],
                        quantum_register[
                            l_index
                        ],
                    )
                else:
                    circuit.cx(
                        i_index,
                        l_index,
                    )

                    circuit.append(
                        CCY_parameterized(
                            identifier
                        ),
                        [
                            quantum_register[
                                i_index
                            ],
                            quantum_register[
                                i_index + 1
                            ],
                            quantum_register[
                                l_index
                            ],
                        ],
                    )

                    circuit.cx(
                        i_index,
                        l_index,
                    )

            auxiliary_counter += 1

    return circuit


# A mesma semente deve ser usada toda vez que o ansatz é reconstruído,
# pois os nomes aleatórios influenciam a ordenação dos parâmetros.
np.random.seed(
    PARAMETER_NAME_SEED
)

ansatz_base = DickeStateAnsatz(
    n=ising.num_qubits,
    k=K,
).decompose()

PARAMETER_ORDER = list(
    ansatz_base.parameters
)

N_THETAS = len(
    PARAMETER_ORDER
)

if N_THETAS != 30:
    raise RuntimeError(
        "O ansatz reconstruído deveria possuir 30 parâmetros, "
        f"mas possui {N_THETAS}."
    )

print(
    "Qubits do ansatz:",
    ansatz_base.num_qubits,
)

print(
    "Parâmetros do ansatz:",
    N_THETAS,
)


## 6. Confirmar o significado dos índices

O índice salvo no banco deve corresponder ao mesmo parâmetro usado por `assign_parameters`.

A tabela abaixo registra, quando possível:

- índice;
- nome simbólico;
- porta;
- qubits;
- posição da instrução no circuito.


In [ ]:
# ============================================================
# 7. IDENTIDADE ESTRUTURAL DOS THETA
# ============================================================

print(
    "ORDEM REAL DO CIRCUITO"
)

for index, parameter in enumerate(
    PARAMETER_ORDER
):
    print(
        f"theta_{index:02d} -> "
        f"{parameter.name}"
    )


parameter_to_index = {
    parameter: index
    for index, parameter
    in enumerate(
        PARAMETER_ORDER
    )
}

identity_rows = []

for instruction_position, instruction in enumerate(
    ansatz_base.data
):
    operation = (
        instruction.operation
    )

    qubits = tuple(
        ansatz_base.find_bit(
            qubit
        ).index
        for qubit
        in instruction.qubits
    )

    for expression in operation.params:
        for parameter in getattr(
            expression,
            "parameters",
            set(),
        ):
            if (
                parameter
                not in parameter_to_index
            ):
                continue

            theta_index = (
                parameter_to_index[
                    parameter
                ]
            )

            identity_rows.append(
                {
                    "theta_index": (
                        theta_index
                    ),
                    "theta_name": (
                        f"theta_{theta_index:02d}"
                    ),
                    "parameter_name": (
                        parameter.name
                    ),
                    "gate": (
                        operation.name.upper()
                    ),
                    "qubits": str(
                        qubits
                    ),
                    "instruction_position": (
                        instruction_position
                    ),
                }
            )

theta_identity = pd.DataFrame(
    identity_rows
)

theta_identity.to_csv(
    TABLE_OUTPUT
    / "theta_identity.csv",
    index=False,
)

if theta_identity.empty:
    print(
        "Os parâmetros não ficaram expostos "
        "nas instruções decompostas."
    )
else:
    display(
        theta_identity.loc[
            theta_identity[
                "theta_index"
            ].isin(
                CANDIDATE_THETAS
            )
        ]
    )


## 7. Preparar o banco e auditar a reconstrução

A energia e o bitstring reconstruídos são comparados com as referências salvas.

A análise causal só continua quando a reconstrução representa o mesmo problema.


In [ ]:
# ============================================================
# 8. MATRIZ DOS VETORES E AUDITORIA DO BANCO
# ============================================================

def parse_theta_vector(
    value,
):
    """
    Converte best_parameters em um vetor NumPy.
    """
    if isinstance(
        value,
        np.ndarray,
    ):
        return value.astype(
            float
        ).ravel()

    if isinstance(
        value,
        (
            list,
            tuple,
            pd.Series,
        ),
    ):
        return np.asarray(
            value,
            dtype=float,
        ).ravel()

    if isinstance(
        value,
        str,
    ):
        text = value.strip()

        try:
            parsed = json.loads(
                text
            )
        except Exception:
            parsed = ast.literal_eval(
                text
            )

        return np.asarray(
            parsed,
            dtype=float,
        ).ravel()

    raise TypeError(
        "Tipo inválido em best_parameters: "
        f"{type(value)}"
    )


def wrap_angle(
    values,
):
    """
    Coloca os ângulos em [-pi, pi).
    """
    values = np.asarray(
        values,
        dtype=float,
    )

    return (
        values
        + np.pi
    ) % (
        2
        * np.pi
    ) - np.pi


def parse_saved_optimal_bitstrings(
    value,
):
    """
    Extrai bitstrings de best_answer quando a coluna existe.
    """
    if isinstance(
        value,
        (
            list,
            tuple,
            set,
            np.ndarray,
        ),
    ):
        raw_values = list(
            value
        )
    elif isinstance(
        value,
        str,
    ):
        text = value.strip()

        try:
            parsed = ast.literal_eval(
                text
            )
        except Exception:
            parsed = text

        if isinstance(
            parsed,
            (
                list,
                tuple,
                set,
            ),
        ):
            raw_values = list(
                parsed
            )
        else:
            raw_values = [
                parsed
            ]
    else:
        raw_values = [
            value
        ]

    cleaned = []

    for raw_value in raw_values:
        candidate = str(
            raw_value
        ).replace(
            " ",
            "",
        )

        if (
            len(
                candidate
            )
            == N_ASSETS
            and set(
                candidate
            ).issubset(
                {
                    "0",
                    "1",
                }
            )
        ):
            cleaned.append(
                candidate
            )

    return sorted(
        set(
            cleaned
        )
    )


theta_list = dataset[
    "best_parameters"
].map(
    parse_theta_vector
)

vector_lengths = theta_list.map(
    len
)

if vector_lengths.nunique() != 1:
    raise ValueError(
        "Os vetores theta possuem comprimentos diferentes."
    )

THETA = wrap_angle(
    np.vstack(
        theta_list.to_numpy()
    )
)

ENERGY_SAVED = pd.to_numeric(
    dataset[
        "objective_function_value"
    ],
    errors="coerce",
).to_numpy(
    dtype=float
)

BITSTRING_SAVED = (
    dataset[
        "most_frequent_bitstring"
    ]
    .astype(str)
    .str.replace(
        " ",
        "",
        regex=False,
    )
    .to_numpy()
)

valid_rows = (
    np.all(
        np.isfinite(
            THETA
        ),
        axis=1,
    )
    & np.isfinite(
        ENERGY_SAVED
    )
)

dataset = (
    dataset.loc[
        valid_rows
    ]
    .reset_index(
        drop=True
    )
)

THETA = THETA[
    valid_rows
]

ENERGY_SAVED = ENERGY_SAVED[
    valid_rows
]

BITSTRING_SAVED = BITSTRING_SAVED[
    valid_rows
]

if (
    THETA.shape[1]
    != N_THETAS
):
    raise RuntimeError(
        "O banco possui "
        f"{THETA.shape[1]} parâmetros por vetor, "
        f"mas o ansatz possui {N_THETAS}."
    )


bank_exact_energy = np.nan

if (
    "best_objective_function_value"
    in dataset.columns
):
    bank_exact_values = pd.to_numeric(
        dataset[
            "best_objective_function_value"
        ],
        errors="coerce",
    ).dropna()

    if not bank_exact_values.empty:
        bank_exact_energy = float(
            bank_exact_values.median()
        )


bank_optimal_bitstrings = []

if (
    "best_answer"
    in dataset.columns
):
    for value in dataset[
        "best_answer"
    ].head(
        min(
            100,
            len(
                dataset
            ),
        )
    ):
        bank_optimal_bitstrings.extend(
            parse_saved_optimal_bitstrings(
                value
            )
        )

bank_optimal_bitstrings = sorted(
    set(
        bank_optimal_bitstrings
    )
)


audit_rows = [
    {
        "check": "n_rows",
        "reconstructed": len(
            dataset
        ),
        "bank_reference": (
            "10000 esperado no projeto"
        ),
        "difference": (
            len(
                dataset
            )
            - 10_000
        ),
        "status": (
            "ok"
            if len(
                dataset
            )
            == 10_000
            else "warning"
        ),
    },
    {
        "check": "n_theta",
        "reconstructed": (
            N_THETAS
        ),
        "bank_reference": (
            THETA.shape[1]
        ),
        "difference": (
            N_THETAS
            - THETA.shape[1]
        ),
        "status": (
            "ok"
            if N_THETAS
            == THETA.shape[1]
            else "fail"
        ),
    },
    {
        "check": "exact_bitstring",
        "reconstructed": (
            EXACT_BITSTRING
        ),
        "bank_reference": (
            bank_optimal_bitstrings
            if bank_optimal_bitstrings
            else "não disponível"
        ),
        "difference": "",
        "status": (
            "ok"
            if (
                not bank_optimal_bitstrings
                or EXACT_BITSTRING
                in bank_optimal_bitstrings
            )
            else "fail"
        ),
    },
]

if np.isfinite(
    bank_exact_energy
):
    exact_energy_difference = abs(
        EXACT_ENERGY
        - bank_exact_energy
    )

    audit_rows.append(
        {
            "check": "exact_energy",
            "reconstructed": (
                EXACT_ENERGY
            ),
            "bank_reference": (
                bank_exact_energy
            ),
            "difference": (
                exact_energy_difference
            ),
            "status": (
                "ok"
                if exact_energy_difference
                <= 1e-7
                else "fail"
            ),
        }
    )

reconstruction_audit = pd.DataFrame(
    audit_rows
)

reconstruction_audit.to_csv(
    TABLE_OUTPUT
    / "reconstruction_audit.csv",
    index=False,
)

display(
    reconstruction_audit
)

failed_checks = reconstruction_audit[
    reconstruction_audit[
        "status"
    ]
    == "fail"
]

if (
    STRICT_RECONSTRUCTION
    and not failed_checks.empty
):
    raise RuntimeError(
        "A reconstrução não coincide com o banco original. "
        "Consulte reconstruction_audit.csv antes de continuar."
    )


dataset[
    "energy_gap_saved"
] = np.abs(
    ENERGY_SAVED
    - EXACT_ENERGY
)

dataset[
    "is_optimal_saved"
] = np.isin(
    BITSTRING_SAVED,
    OPTIMAL_BITSTRINGS,
)

print(
    "Formato do banco:",
    THETA.shape,
)

print(
    "Energia exata:",
    EXACT_ENERGY,
)

print(
    "Bitstring(s) ótimo(s):",
    OPTIMAL_BITSTRINGS,
)

print(
    "Taxa ótima salva:",
    f"{100 * dataset['is_optimal_saved'].mean():.2f}%",
)


# 5. Separar descoberta e validação

O split evita descobrir e validar a mesma faixa angular com os mesmos vetores.


In [ ]:

# ============================================================
# 5. SPLIT 70/30
# ============================================================

rng = np.random.default_rng(RANDOM_STATE)
positions = rng.permutation(len(dataset))
cut = int(DISCOVERY_FRACTION * len(dataset))

discovery_positions = positions[:cut]
validation_positions = positions[cut:]

discovery_df = dataset.iloc[discovery_positions].reset_index(drop=True)
validation_df = dataset.iloc[validation_positions].reset_index(drop=True)
THETA_DISCOVERY = THETA[discovery_positions]
THETA_VALIDATION = THETA[validation_positions]

print('Descoberta:', len(discovery_df))
print('Validação:', len(validation_df))


# 6. Descobrir regiões boas, ruins e controles

Para cada \(\theta_j\), o intervalo angular é dividido em 24 faixas. Em cada faixa são calculados o gap mediano e a taxa do bitstring ótimo.

- **boa:** menor gap e maior taxa ótima;
- **ruim:** maior gap e menor taxa ótima;
- **controle:** parâmetro com baixo contraste empírico e cobertura angular suficiente.


In [ ]:

# ============================================================
# 6. REGIÕES ANGULARES
# ============================================================

angle_edges = np.linspace(-np.pi, np.pi, N_ANGLE_BINS + 1)
angle_centers = 0.5 * (angle_edges[:-1] + angle_edges[1:])
region_rows = []

for theta_index in range(N_THETAS):
    values = THETA_DISCOVERY[:, theta_index]
    bin_ids = np.digitize(values, angle_edges, right=False) - 1
    bin_ids = np.clip(bin_ids, 0, N_ANGLE_BINS - 1)

    for bin_id in range(N_ANGLE_BINS):
        local_positions = np.flatnonzero(bin_ids == bin_id)
        if len(local_positions) < MIN_RUNS_PER_BIN:
            continue

        local_df = discovery_df.iloc[local_positions]
        region_rows.append({
            'theta_index': theta_index,
            'theta_name': f'theta_{theta_index:02d}',
            'bin_id': bin_id,
            'angle_center': float(angle_centers[bin_id]),
            'angle_left': float(angle_edges[bin_id]),
            'angle_right': float(angle_edges[bin_id + 1]),
            'n_runs': int(len(local_positions)),
            'median_gap': float(local_df['energy_gap_saved'].median()),
            'optimal_rate': float(local_df['is_optimal_saved'].mean()),
        })

region_table = pd.DataFrame(region_rows)
choice_rows = []

for theta_index, group in region_table.groupby('theta_index', sort=True):
    if len(group) < 2:
        continue

    good = group.sort_values(
        ['median_gap', 'optimal_rate'], ascending=[True, False]
    ).iloc[0]
    bad = group.sort_values(
        ['median_gap', 'optimal_rate'], ascending=[False, True]
    ).iloc[0]

    choice_rows.append({
        'theta_index': int(theta_index),
        'theta_name': f'theta_{int(theta_index):02d}',
        'valid_regions': int(len(group)),
        'good_angle': float(good['angle_center']),
        'good_left': float(good['angle_left']),
        'good_right': float(good['angle_right']),
        'good_median_gap': float(good['median_gap']),
        'good_optimal_rate': float(good['optimal_rate']),
        'good_n': int(good['n_runs']),
        'bad_angle': float(bad['angle_center']),
        'bad_left': float(bad['angle_left']),
        'bad_right': float(bad['angle_right']),
        'bad_median_gap': float(bad['median_gap']),
        'bad_optimal_rate': float(bad['optimal_rate']),
        'bad_n': int(bad['n_runs']),
        'energy_contrast': float(bad['median_gap'] - good['median_gap']),
        'bitstring_contrast': float(good['optimal_rate'] - bad['optimal_rate']),
    })

region_choices = pd.DataFrame(choice_rows)
for theta_index in CANDIDATE_THETAS:
    if theta_index not in set(region_choices['theta_index']):
        raise RuntimeError(f'theta_{theta_index:02d} não possui regiões suficientes.')

energy_scale = max(region_choices['energy_contrast'].max(), 1e-12)
bitstring_scale = max(region_choices['bitstring_contrast'].max(), 1e-12)
region_choices['combined_contrast'] = (
    0.5 * region_choices['energy_contrast'] / energy_scale
    + 0.5 * region_choices['bitstring_contrast'] / bitstring_scale
)

control_table = (
    region_choices.loc[
        ~region_choices['theta_index'].isin(CANDIDATE_THETAS)
        & (region_choices['valid_regions'] >= 3)
    ]
    .sort_values('combined_contrast', ascending=True)
    .head(N_CONTROL_THETAS)
)
CONTROL_THETAS = control_table['theta_index'].astype(int).tolist()
TEST_THETAS = CANDIDATE_THETAS + CONTROL_THETAS

region_table.to_csv(TABLE_OUTPUT / 'all_angular_regions.csv', index=False)
region_choices.to_csv(TABLE_OUTPUT / 'good_bad_regions.csv', index=False)

print('Candidatos:', CANDIDATE_THETAS)
print('Controles:', CONTROL_THETAS)
display(region_choices.loc[region_choices['theta_index'].isin(TEST_THETAS)])


# 7. Avaliador exato

A função liga explicitamente `theta_vector[j]` ao parâmetro `PARAMETER_ORDER[j]`, constrói o `Statevector` e mede energia, probabilidade ótima e bitstring dominante. Não há ruído de shots nesta etapa.


In [ ]:

# ============================================================
# 7. AVALIAÇÃO EXATA SEM COBYLA E SEM SHOTS
# ============================================================

def evaluate_theta_exact(theta_vector):
    theta_vector = wrap_angle(np.asarray(theta_vector, dtype=float))
    if len(theta_vector) != N_THETAS:
        raise ValueError(f'O vetor possui {len(theta_vector)} valores.')

    # Ligação explícita: o índice do banco é o índice usado no circuito.
    parameter_map = {
        parameter: theta_vector[index]
        for index, parameter in enumerate(PARAMETER_ORDER)
    }
    bound_circuit = ansatz_base.assign_parameters(parameter_map, inplace=False)
    state = Statevector.from_instruction(bound_circuit)

    energy = float(
        np.real(state.expectation_value(ising)) + float(np.real(offset))
    )
    probabilities = state.probabilities_dict()
    p_optimal = float(sum(
        probabilities.get(bitstring, 0.0)
        for bitstring in OPTIMAL_BITSTRINGS
    ))
    dominant_bitstring = max(probabilities, key=probabilities.get)

    return {
        'energy': energy,
        'energy_gap': abs(energy - EXACT_ENERGY),
        'p_optimal': p_optimal,
        'dominant_bitstring': dominant_bitstring,
        'dominant_probability': float(probabilities[dominant_bitstring]),
        'dominant_is_optimal': dominant_bitstring in OPTIMAL_BITSTRINGS,
    }

quick_test = evaluate_theta_exact(THETA_VALIDATION[0])
quick_test


# 8. Confirmar vetores ótimos e subótimos

A classificação salva veio de 4.096 shots. Antes da intervenção, cada caso é reavaliado pelo `Statevector`. Só entram no teste vetores cuja família é confirmada exatamente.


In [ ]:

# ============================================================
# 8. CASOS CONFIRMADOS
# ============================================================

saved_optimal_mask = validation_df['is_optimal_saved'].to_numpy(dtype=bool)
optimal_positions = np.flatnonzero(saved_optimal_mask)
suboptimal_positions = np.flatnonzero(~saved_optimal_mask)


def collect_confirmed_cases(candidate_positions, want_optimal, n_cases):
    selected = []
    for position in rng.permutation(candidate_positions):
        theta_vector = THETA_VALIDATION[position].copy()
        metrics = evaluate_theta_exact(theta_vector)
        correct_family = (
            metrics['dominant_is_optimal']
            if want_optimal
            else not metrics['dominant_is_optimal']
        )
        if not correct_family:
            continue
        selected.append({
            'position': int(position),
            'theta': theta_vector,
            'base': metrics,
        })
        if len(selected) >= n_cases:
            break
    return selected

optimal_cases = collect_confirmed_cases(
    optimal_positions, want_optimal=True, n_cases=N_TEST_VECTORS
)
suboptimal_cases = collect_confirmed_cases(
    suboptimal_positions, want_optimal=False, n_cases=N_TEST_VECTORS
)

if len(optimal_cases) < N_TEST_VECTORS:
    raise RuntimeError(f'Vetores ótimos confirmados: {len(optimal_cases)}.')
if len(suboptimal_cases) < N_TEST_VECTORS:
    raise RuntimeError(f'Vetores subótimos confirmados: {len(suboptimal_cases)}.')

print('Ótimos confirmados:', len(optimal_cases))
print('Subótimos confirmados:', len(suboptimal_cases))


# 9. Intervenções individuais e conjunta

A função `intervene_single` copia o vetor e modifica somente um índice. Portanto, os outros 29 parâmetros permanecem rigorosamente fixos.


In [ ]:

# ============================================================
# 9. EXECUÇÃO DAS INTERVENÇÕES
# ============================================================

region_lookup = region_choices.set_index('theta_index')


def intervene_single(theta_vector, theta_index, new_angle):
    """Altera somente um theta; os outros 29 permanecem fixos."""
    modified = theta_vector.copy()
    modified[theta_index] = new_angle
    return wrap_angle(modified)


def intervene_group(theta_vector, theta_indices, region_name):
    """Altera simultaneamente o grupo para as regiões good ou bad."""
    if region_name not in {'good', 'bad'}:
        raise ValueError("region_name deve ser 'good' ou 'bad'.")
    modified = theta_vector.copy()
    column = 'good_angle' if region_name == 'good' else 'bad_angle'
    for theta_index in theta_indices:
        modified[theta_index] = float(region_lookup.loc[theta_index, column])
    return wrap_angle(modified)


intervention_rows = []

for theta_index in TEST_THETAS:
    good_angle = float(region_lookup.loc[theta_index, 'good_angle'])
    bad_angle = float(region_lookup.loc[theta_index, 'bad_angle'])
    theta_type = 'candidato' if theta_index in CANDIDATE_THETAS else 'controle'

    # Necessidade: ótimo -> região ruim.
    for case_id, case in enumerate(optimal_cases):
        new_metrics = evaluate_theta_exact(
            intervene_single(case['theta'], theta_index, bad_angle)
        )
        base = case['base']
        intervention_rows.append({
            'test': 'necessidade',
            'theta_index': theta_index,
            'theta_label': f'theta_{theta_index:02d}',
            'theta_type': theta_type,
            'case_id': case_id,
            'base_energy': base['energy'],
            'new_energy': new_metrics['energy'],
            'delta_energy': new_metrics['energy'] - base['energy'],
            'base_p_optimal': base['p_optimal'],
            'new_p_optimal': new_metrics['p_optimal'],
            'delta_p_optimal': new_metrics['p_optimal'] - base['p_optimal'],
            'base_is_optimal': base['dominant_is_optimal'],
            'new_is_optimal': new_metrics['dominant_is_optimal'],
            'base_bitstring': base['dominant_bitstring'],
            'new_bitstring': new_metrics['dominant_bitstring'],
        })

    # Suficiência: subótimo -> região boa.
    for case_id, case in enumerate(suboptimal_cases):
        new_metrics = evaluate_theta_exact(
            intervene_single(case['theta'], theta_index, good_angle)
        )
        base = case['base']
        intervention_rows.append({
            'test': 'suficiencia',
            'theta_index': theta_index,
            'theta_label': f'theta_{theta_index:02d}',
            'theta_type': theta_type,
            'case_id': case_id,
            'base_energy': base['energy'],
            'new_energy': new_metrics['energy'],
            'delta_energy': new_metrics['energy'] - base['energy'],
            'base_p_optimal': base['p_optimal'],
            'new_p_optimal': new_metrics['p_optimal'],
            'delta_p_optimal': new_metrics['p_optimal'] - base['p_optimal'],
            'base_is_optimal': base['dominant_is_optimal'],
            'new_is_optimal': new_metrics['dominant_is_optimal'],
            'base_bitstring': base['dominant_bitstring'],
            'new_bitstring': new_metrics['dominant_bitstring'],
        })

GROUP_LABEL = 'grupo_02_03_14_17'

for test_name, cases, region_name in [
    ('necessidade', optimal_cases, 'bad'),
    ('suficiencia', suboptimal_cases, 'good'),
]:
    for case_id, case in enumerate(cases):
        new_metrics = evaluate_theta_exact(
            intervene_group(case['theta'], CANDIDATE_THETAS, region_name)
        )
        base = case['base']
        intervention_rows.append({
            'test': test_name,
            'theta_index': -1,
            'theta_label': GROUP_LABEL,
            'theta_type': 'grupo',
            'case_id': case_id,
            'base_energy': base['energy'],
            'new_energy': new_metrics['energy'],
            'delta_energy': new_metrics['energy'] - base['energy'],
            'base_p_optimal': base['p_optimal'],
            'new_p_optimal': new_metrics['p_optimal'],
            'delta_p_optimal': new_metrics['p_optimal'] - base['p_optimal'],
            'base_is_optimal': base['dominant_is_optimal'],
            'new_is_optimal': new_metrics['dominant_is_optimal'],
            'base_bitstring': base['dominant_bitstring'],
            'new_bitstring': new_metrics['dominant_bitstring'],
        })

intervention_results = pd.DataFrame(intervention_rows)
intervention_results.to_csv(TABLE_OUTPUT / 'intervention_results.csv', index=False)
print('Intervenções concluídas:', len(intervention_results))
display(intervention_results.head())


# 10. Resumo estatístico

O teste de Wilcoxon é pareado: compara cada vetor antes e depois da intervenção. A taxa de direção mostra em quantos casos a mudança ocorreu no sentido esperado.


In [ ]:

# ============================================================
# 10. RESUMO E TESTE PAREADO
# ============================================================

def safe_wilcoxon(values):
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return np.nan
    if np.allclose(values, 0.0):
        return 1.0
    try:
        return float(wilcoxon(values).pvalue)
    except Exception:
        return np.nan

summary_rows = []

for (test_name, theta_label, theta_type), group in intervention_results.groupby(
    ['test', 'theta_label', 'theta_type'], sort=False
):
    delta_energy = group['delta_energy'].to_numpy(dtype=float)
    delta_probability = group['delta_p_optimal'].to_numpy(dtype=float)

    if test_name == 'necessidade':
        expected_energy_rate = float(np.mean(delta_energy > 0))
        expected_probability_rate = float(np.mean(delta_probability < 0))
        bitstring_success_rate = float(np.mean(~group['new_is_optimal'].to_numpy(bool)))
    else:
        expected_energy_rate = float(np.mean(delta_energy < 0))
        expected_probability_rate = float(np.mean(delta_probability > 0))
        bitstring_success_rate = float(np.mean(group['new_is_optimal'].to_numpy(bool)))

    if expected_energy_rate >= 0.70 and expected_probability_rate >= 0.70:
        conclusion = 'efeito consistente'
    elif expected_energy_rate >= 0.60 or expected_probability_rate >= 0.60:
        conclusion = 'efeito parcial'
    else:
        conclusion = 'associação não confirmada causalmente'

    summary_rows.append({
        'test': test_name,
        'theta_label': theta_label,
        'theta_type': theta_type,
        'n_vectors': int(len(group)),
        'median_delta_energy': float(np.median(delta_energy)),
        'median_delta_p_optimal': float(np.median(delta_probability)),
        'expected_energy_rate': expected_energy_rate,
        'expected_probability_rate': expected_probability_rate,
        'bitstring_success_rate': bitstring_success_rate,
        'p_value_energy': safe_wilcoxon(delta_energy),
        'p_value_probability': safe_wilcoxon(delta_probability),
        'automatic_conclusion': conclusion,
    })

causal_summary = pd.DataFrame(summary_rows)
causal_summary.to_csv(TABLE_OUTPUT / 'causal_summary.csv', index=False)
display(causal_summary.sort_values(['test', 'theta_type', 'theta_label']))


# 11. Gráficos autocomentados

- necessidade: resultado esperado \(\Delta E>0\) e \(\Delta P^\star<0\);
- suficiência: resultado esperado \(\Delta E<0\) e \(\Delta P^\star>0\).


In [ ]:

# ============================================================
# 11. FUNÇÃO ÚNICA PARA OS QUATRO GRÁFICOS
# ============================================================

def plot_causal_summary(test_name, metric, filename):
    table = causal_summary.loc[causal_summary['test'] == test_name].copy()

    if metric == 'energy':
        column = 'median_delta_energy'
        factor = 1.0
        xlabel = 'ΔE mediano = valor modificado − valor original'
        expected = 'ΔE > 0' if test_name == 'necessidade' else 'ΔE < 0'
    else:
        column = 'median_delta_p_optimal'
        factor = 100.0
        xlabel = 'ΔP(bitstring ótimo), em pontos percentuais'
        expected = 'ΔP* < 0' if test_name == 'necessidade' else 'ΔP* > 0'

    table['plot_value'] = factor * table[column]
    table = table.sort_values('plot_value', ascending=True)

    fig, ax = plt.subplots(figsize=(12, 7))
    bars = ax.barh(table['theta_label'], table['plot_value'])
    ax.axvline(0.0, linestyle='--', linewidth=1.5)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Intervenção')
    ax.set_title(f'{test_name.capitalize()} — {metric}\nResultado esperado: {expected}')
    ax.grid(axis='x', alpha=0.25)

    for bar, value in zip(bars, table['plot_value']):
        suffix = ' p.p.' if metric == 'probability' else ''
        ax.text(value, bar.get_y() + bar.get_height()/2, f' {value:.4f}{suffix}', va='center')

    # Escolhe o maior efeito na direção esperada.
    if test_name == 'necessidade' and metric == 'energy':
        strongest = table.sort_values('plot_value', ascending=False).iloc[0]
    elif test_name == 'necessidade' and metric == 'probability':
        strongest = table.sort_values('plot_value', ascending=True).iloc[0]
    elif test_name == 'suficiencia' and metric == 'energy':
        strongest = table.sort_values('plot_value', ascending=True).iloc[0]
    else:
        strongest = table.sort_values('plot_value', ascending=False).iloc[0]

    ax.text(
        0.98, 0.05,
        'LEITURA AUTOMÁTICA\n'
        f"Maior efeito: {strongest['theta_label']}\n"
        f"Valor: {strongest['plot_value']:.4f}\n"
        f"Conclusão: {strongest['automatic_conclusion']}",
        transform=ax.transAxes,
        ha='right', va='bottom',
        bbox={'boxstyle': 'round,pad=0.5', 'facecolor': 'white', 'alpha': 0.92},
    )

    fig.tight_layout()
    fig.savefig(FIGURE_OUTPUT / filename, dpi=300, bbox_inches='tight')
    plt.show()


plot_causal_summary('necessidade', 'energy', '01_necessity_energy.png')
plot_causal_summary('necessidade', 'probability', '02_necessity_probability.png')
plot_causal_summary('suficiencia', 'energy', '03_sufficiency_energy.png')
plot_causal_summary('suficiencia', 'probability', '04_sufficiency_probability.png')


# 12. Relatório final

A conclusão mais forte ocorre quando:

1. candidatos superam os controles;
2. necessidade e suficiência apontam na direção esperada;
3. a intervenção conjunta supera as individuais.


In [ ]:

# ============================================================
# 12. CONCLUSÃO AUTOMÁTICA
# ============================================================

print('=' * 82)
print('VALIDAÇÃO CAUSAL DOS THETA')
print('=' * 82)

for theta_index in CANDIDATE_THETAS:
    label = f'theta_{theta_index:02d}'
    print('\n' + label)
    print('-' * len(label))

    for test_name in ['necessidade', 'suficiencia']:
        row = causal_summary.loc[
            (causal_summary['theta_label'] == label)
            & (causal_summary['test'] == test_name)
        ]
        if row.empty:
            continue
        row = row.iloc[0]
        print(f"{test_name.capitalize()}: {row['automatic_conclusion']}")
        print(f"  ΔE mediano: {row['median_delta_energy']:.6f}")
        print(f"  ΔP ótimo: {100 * row['median_delta_p_optimal']:.1f} p.p.")
        print(f"  Direção energética esperada: {100 * row['expected_energy_rate']:.1f}%")
        print(f"  Mudança correta do dominante: {100 * row['bitstring_success_rate']:.1f}%")

print('\nINTERVENÇÃO CONJUNTA')
display(causal_summary.loc[causal_summary['theta_label'] == GROUP_LABEL])

control_labels = [f'theta_{index:02d}' for index in CONTROL_THETAS]
print('\nCONTROLES NEGATIVOS')
display(causal_summary.loc[causal_summary['theta_label'].isin(control_labels)])

print('\nTabelas:', TABLE_OUTPUT.resolve())
print('Figuras:', FIGURE_OUTPUT.resolve())


## Independência confirmada

Este notebook contém internamente:

- leitura do CSV;
- reconstrução de retornos e covariância;
- CVXPY;
- enumeração clássica;
- programa quadrático;
- QUBO;
- Ising;
- solução exata;
- `CY_parameterized`;
- `CCY_parameterized`;
- `DickeStateAnsatz`;
- ordem dos 30 parâmetros;
- análise causal.

O único arquivo externo de resultados necessário é o banco já salvo, como `merged.pkl`.

Nenhuma das 10.000 otimizações é refeita.
